### Part 0: Installing libraries

In [1]:
%pip install spacy

In [2]:
%pip install spacy-universal-sentence-encoder

  Preparing metadata (setup.py) ... done
  Created wheel for spacy-universal-sentence-encoder: filename=spacy_universal_sentence_encoder-0.4.6-py3-none-any.whl size=16541 sha256=88b2b2ab73f45cd81777828d19739451043a79356a87731929b64f952ff2ccf7
  Stored in directory: /root/.cache/pip/wheels/b8/c5/91/dc7e03329421578e554c8d2635f92fd528531123cce102c7f2
Successfully built spacy-universal-sentence-encoder


In [3]:
%pip install pyvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 46.5 MB/s eta 0:00:00


In [4]:
%pip install geopy

### Part 1: Constructing the similarity index

In order to construct the similarity index, we use spacy, in particular a universal encoder which allows for a more accurate word embedding process. This will help us in detecting true similarity, and getting rid of false-similar sentences, such as sentences that include a large amount of stop-words (e.g sentences that use "the", "a", "he", "it", etc. a lot will not appear as similar, but rather the true meaning of the sentence will be analysed).

In [5]:
import spacy
import numpy as np
import pandas as pd
import networkx as nx
import spacy_universal_sentence_encoder
from tqdm import tqdm
from google.colab import files
from pyvis.network import Network
from IPython.display import display, HTML
import folium
from geopy.geocoders import Nominatim

In [6]:
nlp = spacy_universal_sentence_encoder.load_model('en_use_lg')

Downloaded https://tfhub.dev/google/universal-sentence-encoder-large/5, Total size: 577.10MB



We read the data from the reporting frameworks table that we are working on.

In [26]:
reporting_data = pd.read_excel('ReportingFrameworks.xlsx')

In [27]:
reporting_data

,Framework,Topic,Reference,Recommendation,G_Ref,Scope,Guidance
0,TCFD,Governance,TCFD Governance A,Describe the board’s oversight of climate-rela...,TCFD Governance A1,Generic,Consider including a discussion of the followi...
1,TCFD,Governance,TCFD Governance B,Describe management’s role in assessing and ma...,TCFD Governance B1,Generic,Consider including the following information:\...
2,TCFD,Strategy,TCFD Strategy A,Describe the climate-related risks and opportu...,TCFD Strategy A1,Generic,Organisations should provide the following inf...
3,TCFD,Strategy,TCFD Strategy A,Describe the climate-related risks and opportu...,TCFD Strategy A4,Generic,Organizations should consider providing a desc...
4,TCFD,Strategy,TCFD Strategy A,Describe the climate-related risks and opportu...,TCFD Strategy A5,Banks,Banks should describe significant concentratio...
...,...,...,...,...,...,...,...
695,SBTi,MetricsandTargets,FINZ-C19 C19.1,"Financial institutions shall adhere, at all ti...",NaN,Financial Institutions,SBTi CLAIMS
696,SBTi,MetricsandTargets,FINZ-C19 C19.2,Claims made by financial institutions shall ac...,NaN,Financial Institutions,SBTi CLAIMS
697,SBTi,MetricsandTargets,FINZ-C19 C19.3,Financial institutions shall ensure that all c...,NaN,Financial Institutions,SBTi CLAIMS
698,SBTi,MetricsandTargets,FINZ-C19 C19.4,All claim content shall be fully substantiated...,NaN,Financial Institutions,SBTi CLAIMS


In [28]:
def similarity_finder(data, columns, run_aggregation = True):
  """
  This is a function that is calculating the similarity index between all of the entries in a specific column, using the spacy framework.

  Parameters:
  "data" represents the data that we are putting in the format provided to us by Lloyd.
  "columns" contains, in order:
  - the column which contains the framework
  - the column which contains the topic
  - the column which contains the exact reference
  - the column which contains the entry that we want to compare (which will initially be the recommendation, but may become the guidance)

  Outputs:
  The output will be similar to the similarrequrements.csv file. This output will then be fed into a download function, in case we want to download the file,
  and a networkx code which will create a network of all of these links.
  """
  framework_column = columns[0]
  topic_column = columns[1]
  reference_column = columns[2]
  comparison_column = columns[3]

  if(run_aggregation):
    data = data.replace("ESRS E1", "ESRS")
    data = data.replace("ESRS E4", "ESRS")
    data = data.replace("IFRS S1", "IFRS")
    data = data.replace("IFRS S2", "IFRS")

  n = int(len(data)*(len(data)-1)/2)
  similarity_list = []

  # Create a progress bar for the total number of comparisons, while also creating the similarity table file
  with tqdm(total=n, desc="Comparing pairs") as pbar:
    for i in range(len(data)):
      curr_framework = data.iloc[i][framework_column]
      curr_topic = data.iloc[i][topic_column]
      curr_reference = data.iloc[i][reference_column]
      curr_comparison = data.iloc[i][comparison_column]
      curr_nlp = nlp(curr_comparison)
      for j in range(i+1, len(data)):
        next_framework = data.iloc[j][framework_column]
        next_topic = data.iloc[j][topic_column]
        next_reference = data.iloc[j][reference_column]
        next_comparison = data.iloc[j][comparison_column]
        next_nlp = nlp(next_comparison)
        if (next_framework != curr_framework) and (next_topic == curr_topic): #We are getting rid of the similarities within the same framework, which are bound to be high, and we only calculate similarities where they are useful, in the same topic
          similarity_score = curr_nlp.similarity(next_nlp)
          similarity_list.append({"Topic":curr_topic, "Framework 1": curr_framework, "Reference 1": curr_reference, "Recommendation 1": curr_comparison, "Framework 2": next_framework, "Reference 2": next_reference, "Recommendation 2": next_comparison, "Similarity": similarity_score})
        pbar.update(1)  # Update the progress bar after each comparison

  similarity_table = pd.DataFrame(similarity_list)
  similarity_table.drop_duplicates(inplace=True)
  similarity_table.reset_index(drop=True, inplace=True)
  return similarity_table

In [29]:
similarity_table = similarity_finder(reporting_data, ["Framework", "Topic", "Reference", "Recommendation"])

Comparing pairs: 100%|██████████| 244650/244650 [01:52<00:00, 2171.10it/s]


In [30]:
def download_table(table, filename):
  """
  Downloads tables as a CSV file in Google Colab.

  Parameters:
  - table: pandas DataFrame to download
  - filename: name of the file to save (without extension)

  """
  # Save the DataFrame to a CSV file
  table.to_csv(filename, index=False)

  # Download the file
  files.download(filename)

  print(f"File '{filename}' has been downloaded!")

In [12]:
download_table(similarity_table, "similarity_table.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File 'similarity_table.csv' has been downloaded!


### Part 2: Constructing the network of similarity

Below, what we will be creating is a similarity metric between different frameworks. The list of topics from which one can choose below is as follows: Disclosure, Governance, Metrics and Targets, Risk Management and Strategy.

In [31]:
def similarity_network_table(original_list = ["Disclosure", "Governance", "MetricsandTargets", "RiskManagement", "Strategy"], original_data=similarity_table):
  """
  This is a function that is calculating the data required to build a network of the similarities of different frameworks, based on a list of topics.

  Parameters:
  "list" is the list of topics on which we want to base the similarity. It can contain all topics, but it must include at least one. It's originally defined as the list of all topics.
  "data" is the underlying data, which in this file should be the similarity table found before. It's originally defined as the similarity_table. We are assuming that the data is in the format of the similarity table.

  Outputs:
  A table containing the similarity between all of the frameworks, based on the list of topics. This similarity is calculated as the average of the similarities of all recommendations
  in the chosen topics.
  """
  filtered_data = original_data.copy()
  for i in range(len(filtered_data)):
    if not(filtered_data["Topic"][i] in original_list):
      filtered_data.drop(i, inplace=True)
  filtered_data.reset_index(drop=True, inplace=True)
  framework_1 = pd.unique(filtered_data["Framework 1"])
  framework_2 = pd.unique(filtered_data["Framework 2"])
  all_frameworks = np.concatenate((framework_1, framework_2))
  all_frameworks = pd.unique(all_frameworks)
  similarity_list = []
  with tqdm(total=len(all_frameworks)*(len(all_frameworks)-1)/2, desc="Comparing pairs") as pbar:
    for i in range(len(all_frameworks)):
      curr_framework = all_frameworks[i]
      for j in range(i+1,len(all_frameworks)):
        next_framework = all_frameworks[j]
        similarity = filtered_data.loc[(filtered_data["Framework 1"] == curr_framework) & (filtered_data["Framework 2"] == next_framework)]["Similarity"].mean()
        similarity_list.append({"Framework 1": curr_framework, "Framework 2": next_framework, "Similarity": similarity})
        pbar.update(1)

  similarity_network = pd.DataFrame(similarity_list)
  similarity_network.fillna(0, inplace=True)
  similarity_network.drop_duplicates(inplace=True)
  similarity_network.reset_index(drop=True, inplace=True)
  return similarity_network

In [32]:
similarity_network_table_all_metrics = similarity_network_table()
similarity_network_table_disclosure = similarity_network_table(["Disclosure"])
similarity_network_table_governance = similarity_network_table(["Governance"])
similarity_network_table_metrics = similarity_network_table(["MetricsandTargets"])
similarity_network_table_risk = similarity_network_table(["RiskManagement"])
similarity_network_table_strategy = similarity_network_table(["Strategy"])

Comparing pairs: 100%|██████████| 45/45.0 [00:00<00:00, 338.75it/s]
Comparing pairs: 100%|██████████| 10/10.0 [00:00<00:00, 1017.44it/s]
Comparing pairs: 100%|██████████| 36/36.0 [00:00<00:00, 1255.41it/s]
Comparing pairs: 100%|██████████| 6/6.0 [00:00<00:00, 461.53it/s]
Comparing pairs: 100%|██████████| 28/28.0 [00:00<00:00, 717.44it/s]
Comparing pairs: 100%|██████████| 15/15.0 [00:00<00:00, 781.64it/s]


In [33]:
similarity_table

,Topic,Framework 1,Reference 1,Recommendation 1,Framework 2,Reference 2,Recommendation 2,Similarity
0,Governance,TCFD,TCFD Governance A,Describe the board’s oversight of climate-rela...,TNFD,TNFD Governance A,Describe the board’s oversight of naturerelate...,0.861571
1,Governance,TCFD,TCFD Governance A,Describe the board’s oversight of climate-rela...,TNFD,TNFD Governance B,Describe management’s role in assessing and ma...,0.669910
2,Governance,TCFD,TCFD Governance A,Describe the board’s oversight of climate-rela...,TNFD,TNFD Governance C,Describe the organisation’s human rights polic...,0.429584
3,Governance,TCFD,TCFD Governance A,Describe the board’s oversight of climate-rela...,IFRS,IFRS S1 26,The objective of sustainability-related financ...,0.268225
4,Governance,TCFD,TCFD Governance A,Describe the board’s oversight of climate-rela...,IFRS,IFRS S1 27,"To achieve this objective, an entity shall dis...",0.210691
...,...,...,...,...,...,...,...,...
11139,Disclosure,OSFI,OSFI Chapter 2 Principle 6,The company should disclose information consis...,SBTi,FINZ-C17 C17.1,Financial institutions shall commit to separat...,0.115417
11140,Disclosure,OSFI,OSFI Chapter 2 Principle 6,The company should disclose information consis...,SBTi,FINZ-C17 C17.2,Financial institutions shall commit to startin...,0.169681
11141,Disclosure,OSFI,OSFI Chapter 2 Principle 6,The company should disclose information consis...,SBTi,FINZ-C17 R17.1,Financial institutions should annually and pub...,0.191425
11142,Disclosure,OSFI,OSFI Chapter 2 Principle 6,The company should disclose information consis...,SBTi,FINZ-C17 R17.2,Financial institutions should use data with th...,0.257425


We are now going to asign to each framework a country where it either originated from or is in use.

In [34]:
download_table(similarity_network_table_all_metrics, "similarity_network_table_all_metrics.csv")
download_table(similarity_network_table_disclosure, "similarity_network_table_disclosure.csv")
download_table(similarity_network_table_governance, "similarity_network_table_governance.csv")
download_table(similarity_network_table_metrics, "similarity_network_table_metrics.csv")
download_table(similarity_network_table_risk, "similarity_network_table_risk.csv")
download_table(similarity_network_table_strategy, "similarity_network_table_strategy.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File 'similarity_network_table_all_metrics.csv' has been downloaded!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

File 'similarity_network_table_risk.csv' has been downloaded!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File 'similarity_network_table_strategy.csv' has been downloaded!


### Map of regulations and adopting countries

In [16]:
adoption_dict = {
    "Frameworks":["TCFD", "TNFD", "PRA", "IFRS", "TPT", "BMA", "MAS", "ESRS", "OSFI", "SBTI"],
    "Countries":[["Canada", "France", "Germany", "Italy", "Japan", "United Kingdom", "USA", "New Zealand", "Switzerland", "Singapore", "Brazil", "China", "South Africa"],
                 ["Brazil", "China", "Colombia", "Costa Rica", "Egypt", "India", "Indonesia", "Kenya", "Malaysia", "Mexico", "Morocco", "Nigeria", "Peru", "Philippines", "South Africa"],
                 ["United Kingdom"],
                 ["Turkey", "Bangladesh", "Brazil", "Australia", "Japan", "United Kingdom", "Canada", "Singapore", "New Zealand", "Nigeria", "South Africa", "Malaysia", "China"],
                 ["United Kingdom"],
                 ["Bermuda"],
                 ["Singapore"],
                 ["Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czech Republic", "Denmark", "Estonia", "Finland", "France", "Germany", "Greece", "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta", "Netherlands", "Poland", "Portugal", "Romania", "Slovakia", "Slovenia", "Spain", "Sweden"],
                 ["Canada"],
                 ["Japan", "United Kingdom", "USA", "China", "Germany", "France", "India", "Italy", "Canada", "South Korea", "Mexico", "Brazil", "Australia", "South Africa", "Turkey", "Romania", "Malta"]]
}

## Framework comparison tool

In [35]:
pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 28.3 MB/s eta 0:00:00


In [36]:
import pymupdf

In [54]:
doc = pymupdf.open("climate-transition-plan-2024.pdf")

In [59]:
paragraph_list = []
text_list = []
for page in doc: # iterate the document pages
  text = page.get_text() # get plain text encoded as UTF-8
  text_list.append(text)
text_list = [text.replace('\n', ' ') for text in text_list]

page 0 of climate-transition-plan-2024.pdf
page 1 of climate-transition-plan-2024.pdf
page 2 of climate-transition-plan-2024.pdf
page 3 of climate-transition-plan-2024.pdf
page 4 of climate-transition-plan-2024.pdf
page 5 of climate-transition-plan-2024.pdf
page 6 of climate-transition-plan-2024.pdf
page 7 of climate-transition-plan-2024.pdf
page 8 of climate-transition-plan-2024.pdf
page 9 of climate-transition-plan-2024.pdf
page 10 of climate-transition-plan-2024.pdf
page 11 of climate-transition-plan-2024.pdf
page 12 of climate-transition-plan-2024.pdf
page 13 of climate-transition-plan-2024.pdf
page 14 of climate-transition-plan-2024.pdf
page 15 of climate-transition-plan-2024.pdf
page 16 of climate-transition-plan-2024.pdf
page 17 of climate-transition-plan-2024.pdf
page 18 of climate-transition-plan-2024.pdf
page 19 of climate-transition-plan-2024.pdf
page 20 of climate-transition-plan-2024.pdf
page 21 of climate-transition-plan-2024.pdf
page 22 of climate-transition-plan-2024.pd

In [68]:
text_data_frame=pd.DataFrame(text_list)
text_data_frame.columns=["Paragraph"]

In [76]:
def document_similarity(data, document, columns, run_aggregation = True):
  """
  This is a function that is calculating the similarity index between all of the entries in two documents, using the spacy framework.

  Parameters:
  "data" represents the data that we are putting in the format provided to us by Lloyd, aka the ReportingFrameworks.xlsx doc
  "columns" contains, in order:
  - the column which contains the framework
  - the column which contains the topic
  - the column which contains the exact reference
  - the column which contains the entry that we want to compare (which will initially be the recommendation, but may become the guidance)
  "document" represents the data extracted, page by page, from the pdf document provided

  Outputs:
  The output will be similar to the similarrequrements.csv file. This output will then be fed into a download function, in case we want to download the file,
  and a networkx code which will create a network of all of these links.
  """
  framework_column = columns[0]
  topic_column = columns[1]
  reference_column = columns[2]
  comparison_column = columns[3]

  if(run_aggregation):
    data = data.replace("ESRS E1", "ESRS")
    data = data.replace("ESRS E4", "ESRS")
    data = data.replace("IFRS S1", "IFRS")
    data = data.replace("IFRS S2", "IFRS")

  n = int(len(data)*len(document))
  similarity_list = []


  for i in range(len(data)):
    curr_framework = data.iloc[i][framework_column]
    curr_topic = data.iloc[i][topic_column]
    curr_reference = data.iloc[i][reference_column]
    curr_comparison = data.iloc[i][comparison_column]
    curr_nlp = nlp(curr_comparison)
    for j in range(len(document)):
      doc_comparison = document.iloc[j]['Paragraph'] # Fixed: Changed data.iloc[j] to document.iloc[j]
      doc_nlp = nlp(doc_comparison)
      similarity_score = curr_nlp.similarity(doc_nlp)
      similarity_list.append({"Topic":curr_topic, "Framework": curr_framework, "Reference": curr_reference, "Recommendation": curr_comparison, "Document page": doc_comparison, "Similarity": similarity_score})

  similarity_table = pd.DataFrame(similarity_list)
  similarity_table.drop_duplicates(inplace=True)
  similarity_table.reset_index(drop=True, inplace=True)

  #Calculating the similarity for each table for each topic
  similarity_network_list = []
  all_frameworks = pd.unique(similarity_table["Framework"])
  for i in range(len(all_frameworks)):
    curr_framework = all_frameworks[i]
    curr_framework_data = similarity_table.loc[similarity_table["Framework"] == curr_framework]
    curr_framework_data.reset_index(drop=True, inplace=True)
    all_topics = pd.unique(curr_framework_data["Topic"])
    for j in range(len(all_topics)):
      curr_topic = all_topics[j]
      curr_topic_data = curr_framework_data.loc[curr_framework_data["Topic"] == curr_topic]
      curr_topic_data.reset_index(drop=True, inplace=True)
      similarity = curr_topic_data["Similarity"].mean()
      similarity_network_list.append({"Framework": curr_framework, "Topic": curr_topic, "Similarity": similarity})
    similarity_network_list.append({"Framework": curr_framework, "Topic":"Overall", "Similarity":curr_framework_data["Similarity"].mean()})

  similarity_network = pd.DataFrame(similarity_network_list)
  similarity_network.fillna(0, inplace=True)
  similarity_network.drop_duplicates(inplace=True)

  return similarity_network

In [77]:
sim_table = document_similarity(reporting_data, text_data_frame, ["Framework", "Topic", "Reference", "Recommendation"])

In [78]:
sim_table

,Framework,Topic,Similarity
0,TCFD,Governance,0.076484
1,TCFD,Strategy,0.119560
2,TCFD,RiskManagement,0.053880
3,TCFD,MetricsandTargets,0.090671
4,TCFD,Overall,0.085937
5,TNFD,Governance,0.110009
6,TNFD,Strategy,0.134484
7,TNFD,RiskManagement,0.096288
8,TNFD,MetricsandTargets,0.073423
9,TNFD,Overall,0.105242
